### Plot BLASTp scores v. composite scores
### Julian Moran
### 2026-08-28

In [1]:
import boto3
import glob
import logging
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ============================================================
#       Args
# ============================================================

# API endpoints
ENDPOINT_UNIPROT = "https://rest.uniprot.org/uniprotkb/"
ENDPOINT_UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"

# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [3]:
# ============================================================
#       In
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)
df_comp_score

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [ ]:
# ============================================================
#       Annotate with protein sequences
# ============================================================

def get_uniprot_sequence(
    accession: str,
    endpoint_prefix: str
) -> str:
    url = f"{endpoint_prefix}/{accession}.fasta"
    response = requests.get(url)
    response.raise_for_status()

    sequence = "".join(response.text.splitlines()[1:])
    return sequence

def batch_uniprot_sequences(
    accessions: pl.Series,
    endpoint_prefix: str,
) -> pl.DataFrame:
    col_accessions = accessions.name
    accessions = accessions.to_list()
    query = " OR ".join(f"accession:{acc}" for acc in accessions)
    response = requests.get(
        endpoint_prefix,
        params={
            "query": query,
            "format": "fasta",
            "size": len(accessions)
        }
    )
    response.raise_for_status()
    records = response.text.strip().split(">")[1:]
    df_sequences = pl.DataFrame({
        "accession": [
            record.split("\n", 1)[0].split("|")[1]
            for record in records
        ],
        "sequence": [
            "".join(record.split("\n", 1)[1].splitlines())
            for record in records
        ],
    })
    return df_sequences

# Build lookup dict
hs_accessions_unq = df_comp_score.select(pl.col("human_entryId").unique()).to_series()
accessions = hs_accessions_unq[:100]

tmp = batch_uniprot_sequences(
    accessions = accessions,
    endpoint_prefix=ENDPOINT_UNIPROT_SEARCH
)
tmp

accession,sequence
str,str
"""O75628""","""MTLNTEQEAKTPLHRRASTPLPLSPRGHQP…"
"""P05198""","""MPGLSCRFYQHKFPEVEDVVMVNVRSIAEM…"
"""P46734""","""MESPASSQPASMPQSKGKSKRKKDLRISCM…"
"""Q6UW60""","""MRPAPIALWLRLVLALALVRPRAVGWAPVR…"
"""Q8NE63""","""MSTIQSETDCYDIIEVLGKGTFGEVAKGWR…"
…,…
"""C9JSB1""","""MRLKISLLKEPKHQELVSCVGWTTAEELYS…"
"""A0A075B785""","""MAAMAPGGSGSGGGVNPFLSDSDEDDDEVA…"
"""B4DTL6""","""MASSVGNVADSTEPTKRMLSFQGLAELAHR…"
